In [1]:
import pandas as pd
import numpy as np

In [2]:
df_order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [3]:
df_order_reviews["review_title"] = (
    df_order_reviews["review_comment_title"].fillna('').str.strip()
)

df_order_reviews["review_message"] = (
    df_order_reviews["review_comment_message"].fillna('').str.strip()
)

df_order_reviews['review_text'] = (
    df_order_reviews['review_title'].str.strip().fillna('') 
    + ". " +
    df_order_reviews['review_message'].str.strip().fillna('')
).str.strip(" .")

df_order_reviews['count_chars_review_text'] = (
    df_order_reviews['review_text']
    .str.len()
    .fillna(0)
    .astype(int)
)

df_order_reviews['count_words_review_text'] = (
    df_order_reviews['review_text']
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

df_order_reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_title,review_message,review_text,count_chars_review_text,count_words_review_text
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,,,,0,0
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,,,,0,0
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,,,,0,0
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado,36,6
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...,100,15
...,...,...,...,...,...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30,,,,0,0
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42,,,,0,0
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43,,"Excelente mochila, entrega super rápida. Super...","Excelente mochila, entrega super rápida. Super...",67,9
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13,,,,0,0


## Aplicando lógica para remover reviews desnecessárias

In [4]:
generic_reviews = [
    "bom",
    "muito bom",
    "ótimo",
    "otimo",
    "ótima",
    "otima",
    "excelente",
    "perfeito",
    "gostei",
    "adorei",
    "muito obrigado",
    "boa",
    "maravilha",
    "amei",
    "ok",
    "satisfeito",
    "satisfeita",
    "recomendo",
    "médio",
    "medio",
    "ruim",
    "péssimo",
    "pessimo",
    "horrível",
    "horrivel",
    "não gostei",
    "nao gostei",
]

df_order_reviews['normalized_review_text'] = (
    df_order_reviews["review_text"]
    .fillna("")
    .str.lower()
    .str.replace(r"[^a-zA-Zá-úÁ-Úà-ùâ-ûã-õç0-9\s]", "", regex=True)  # Remove pontuação e especiais
    .str.strip()
)

# Definir a lista de condições (em ordem de prioridade)
condicoes = [
    df_order_reviews["count_chars_review_text"] == 0,
    df_order_reviews["count_words_review_text"] <= 1,
    df_order_reviews["normalized_review_text"].isin(generic_reviews)
]

# Definir os motivos correspondentes a cada condição
motivos = ["no_text", "one_word_review", "generic_review"]

# Aplicar o np.select (default é o valor quando NENHUMA condição acima for verdadeira)
df_order_reviews["use_in_rag_reason"] = np.select(
    condicoes, motivos, default="valid"
)

# A coluna booleana 'use_in_rag' fica simples de preencher baseada na razão:
df_order_reviews["use_in_rag"] = (
    df_order_reviews["use_in_rag_reason"] == "valid"
)

df_order_reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_title,review_message,review_text,count_chars_review_text,count_words_review_text,normalized_review_text,use_in_rag_reason,use_in_rag
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,,,,0,0,,no_text,False
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,,,,0,0,,no_text,False
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,,,,0,0,,no_text,False
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado,36,6,recebi bem antes do prazo estipulado,valid,True
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...,100,15,parabéns lojas lannister adorei comprar pela i...,valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30,,,,0,0,,no_text,False
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42,,,,0,0,,no_text,False
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43,,"Excelente mochila, entrega super rápida. Super...","Excelente mochila, entrega super rápida. Super...",67,9,excelente mochila entrega super rápida super r...,valid,True
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13,,,,0,0,,no_text,False


## Filtrar reviews para RAG

In [5]:
df_reviews_rag = df_order_reviews[df_order_reviews["use_in_rag"]].copy()
df_reviews_rag

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_title,review_message,review_text,count_chars_review_text,count_words_review_text,normalized_review_text,use_in_rag_reason,use_in_rag
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado,36,6,recebi bem antes do prazo estipulado,valid,True
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...,100,15,parabéns lojas lannister adorei comprar pela i...,valid,True
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47,recomendo,aparelho eficiente. no site a marca do aparelh...,recomendo. aparelho eficiente. no site a marca...,185,31,recomendo aparelho eficiente no site a marca d...,valid,True
12,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,NaN,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",2018-02-16 00:00:00,2018-02-20 10:52:22,,"Mas um pouco ,travando...pelo valor ta Boa.","Mas um pouco ,travando...pelo valor ta Boa",42,7,mas um pouco travandopelo valor ta boa,valid,True
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23 00:00:00,2018-05-24 03:00:01,Super recomendo,"Vendedor confiável, produto ok e entrega antes...","Super recomendo. Vendedor confiável, produto o...",72,11,super recomendo vendedor confiável produto ok ...,valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99205,98fffa80dc9acbde7388bef1600f3b15,d398e9c82363c12527f71801bf0e6100,4,NaN,para este produto recebi de acordo com a compr...,2017-11-29 00:00:00,2017-11-30 15:52:51,,para este produto recebi de acordo com a compr...,para este produto recebi de acordo com a compr...,58,10,para este produto recebi de acordo com a compr...,valid,True
99208,df5fae90e85354241d5d64a8955b2b09,509b86c65fe4e2ad5b96408cfef9755e,5,NaN,Entregou dentro do prazo. O produto chegou em ...,2018-02-07 00:00:00,2018-02-19 19:47:23,,Entregou dentro do prazo. O produto chegou em ...,Entregou dentro do prazo. O produto chegou em ...,90,13,entregou dentro do prazo o produto chegou em c...,valid,True
99215,a709d176f59bc3af77f4149c96bae357,d5cb12269711bd1eaf7eed8fd32a7c95,3,NaN,"O produto não foi enviado com NF, não existe v...",2018-05-19 00:00:00,2018-05-20 21:51:06,,"O produto não foi enviado com NF, não existe v...","O produto não foi enviado com NF, não existe v...",124,25,o produto não foi enviado com nf não existe ve...,valid,True
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43,,"Excelente mochila, entrega super rápida. Super...","Excelente mochila, entrega super rápida. Super...",67,9,excelente mochila entrega super rápida super r...,valid,True


In [6]:
df_reviews_rag.drop_duplicates(subset=['review_text'], keep='first', inplace=True)

In [7]:
df_reviews_rag[df_reviews_rag['count_words_review_text'] <= 2]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_title,review_message,review_text,count_chars_review_text,count_words_review_text,normalized_review_text,use_in_rag_reason,use_in_rag
37,2170c9452c2216457b36003fe2969fbb,af32671d0b8ce510a94d9111d97371d3,5,NaN,muito boa,2018-03-22 00:00:00,2018-03-30 23:37:25,,muito boa,muito boa,9,2,muito boa,valid,True
159,42ae74a43661be8587b2e2db054179db,0a187dd7f4efe70e8c3b6e3779b3b133,5,NaN,Ótimo custo-benefício,2017-09-14 00:00:00,2017-09-15 14:15:14,,Ótimo custo-benefício,Ótimo custo-benefício,21,2,ótimo custobenefício,valid,True
228,7914fa319f12631679d1704d7dcd17df,06ecfa8dfae87263e05d953aa561603d,5,Top,Recomendo,2018-08-22 00:00:00,2018-08-23 01:51:28,Top,Recomendo,Top. Recomendo,14,2,top recomendo,valid,True
237,eb6d75bedecae7b3dbea11b8530637d8,00b9e0f8f588d0406a3de447eb606970,5,NaN,Excelente produto.,2018-02-17 00:00:00,2018-02-19 16:20:08,,Excelente produto.,Excelente produto,17,2,excelente produto,valid,True
238,593f9b33c3a4c74e37b7b7c49f0cedc4,268762d6a983092f8632e379836806eb,5,NaN,Tudo certo,2017-09-07 00:00:00,2017-09-13 13:51:51,,Tudo certo,Tudo certo,10,2,tudo certo,valid,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98637,9a6614162d285301aa3ef6de4be75265,5741ea1f91b5fbab2bd2dc653a5b5099,5,NaN,Loja responsável,2017-09-29 00:00:00,2017-10-01 21:52:53,,Loja responsável,Loja responsável,16,2,loja responsável,valid,True
98737,de0f21333a5ccf2376360a6d8a950a75,dec95374a1bf2366249bf770e0568a85,5,NaN,Produto original,2017-08-13 00:00:00,2017-08-14 11:56:12,,Produto original,Produto original,16,2,produto original,valid,True
99016,99227c6c6d3ba9aeb023b53b018aa21c,8899dd616aaabe70b170cf4099307f45,5,NaN,LOJA ÓTIMA,2017-07-01 00:00:00,2017-07-04 11:11:50,,LOJA ÓTIMA,LOJA ÓTIMA,10,2,loja ótima,valid,True
99192,0e7bc73fde6782891898ea71443f9904,bd78f91afbb1ecbc6124974c5e813043,4,👍,Aprovado!,2018-07-04 00:00:00,2018-07-05 00:25:13,👍,Aprovado!,👍. Aprovado!,12,2,aprovado,valid,True


In [8]:
df_reviews_rag[["review_title", "review_message", "review_text", "count_chars_review_text", "count_words_review_text"]]

,review_title,review_message,review_text,count_chars_review_text,count_words_review_text
3,,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado,36,6
4,,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...,100,15
9,recomendo,aparelho eficiente. no site a marca do aparelh...,recomendo. aparelho eficiente. no site a marca...,185,31
12,,"Mas um pouco ,travando...pelo valor ta Boa.","Mas um pouco ,travando...pelo valor ta Boa",42,7
15,Super recomendo,"Vendedor confiável, produto ok e entrega antes...","Super recomendo. Vendedor confiável, produto o...",72,11
...,...,...,...,...,...
99205,,para este produto recebi de acordo com a compr...,para este produto recebi de acordo com a compr...,58,10
99208,,Entregou dentro do prazo. O produto chegou em ...,Entregou dentro do prazo. O produto chegou em ...,90,13
99215,,"O produto não foi enviado com NF, não existe v...","O produto não foi enviado com NF, não existe v...",124,25
99221,,"Excelente mochila, entrega super rápida. Super...","Excelente mochila, entrega super rápida. Super...",67,9


In [11]:
df_reviews_rag.to_parquet('../data/processed/df_reviews_rag.parquet', index=False)